In [23]:
from openai import OpenAI
import os
import re
from pathlib import Path

In [24]:
client = OpenAI(
    api_key= os.environ["TOKEN_KEY"],
    base_url="https://api.opentyphoon.ai/v1"
)

In [25]:
i_prompt = """Role: Specialized Data Extraction Expert
You are an expert in parsing Thai Election Report OCR text and converting it into highly structured data formats. Your objective is to process raw OCR text and output exactly two specific CSV tables.

1. Data Identification & Analysis
Type "บช" (Party List): Identified by keywords like "แบบบัญชีรายชื่อ" or "(บช)".
Type "เขต" (Constituency): Identified by keywords like "แบบแบ่งเขตเลือกตั้ง".

Location Data: Extract names following these keywords:
จังหวัด (Province)
อำเภอ (District)
ตำบล (Sub-district)
หน่วยที่ (Unit Number)

2. Strict Extraction & Formatting Rules
Value Integrity:
- Convert Thai numerals (๐-๙) to Arabic numerals (0-9) before extraction. (thai_digits = "๐๑๒๓๔๕๖๗๘๙")
- If a numeric field is missing, unreadable, or invalid → use -1
- If explicitly zero → use 0
- Prioritize Arabic numerals

Location Cleanup:
- Remove prefixes (จังหวัด, อำเภอ, ตำบล, หน่วยที่)

Language: Keep Thai names

3. Output Formats

CSV 1:
type,province,district,sub-district,unit_number,name,score

CSV 2:
type,province,district,sub-district,unit_number,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots,remain_ballots

Derived Field (remain_ballots):
- remain_ballots = total_ballots - (valid_ballots + invalid_ballots + no_vote_ballots)
- If any value is -1 or result is negative → remain_ballots = -1

Consistency Check:
- Ensure:
  valid_ballots + invalid_ballots + no_vote_ballots + remain_ballots = total_ballots
- If not:
  - Re-check extraction
  - If still inconsistent → assign -1 to unreliable fields

4. CRITICAL:
- Output ONLY two Markdown CSV blocks
- First = CSV1, Second = CSV2
- No explanation
"""

In [ ]:
def generate2csv(
    client,
    district,
    sub_district,
    unit_name,
    province="ลำปาง",
    content="",
    file_name=""
):
    messages = [
        {"role": "system", "content": i_prompt},
        {
            "role": "user",
            "content": f"""จังหวัด: {province}
district: {district}
sub-district: {sub_district}
unit name: {unit_name}

{content}
"""
        }
    ]

    print(f"Processing CSVs: {file_name}")

    response = client.chat.completions.create(
        model="typhoon-v2.5-30b-a3b-instruct",
        messages=messages,
        temperature=0,
        max_tokens=30000,
        top_p=1,
        stream=False
    )

    full_content = response.choices[0].message.content

    print(f"Success CSVs: {file_name}")

    csv_blocks = re.findall(r"```(?:csv)?\s*(.*?)```", full_content, re.DOTALL)

    if len(csv_blocks) < 2:
        parts = full_content.strip().split("\n\n")
        if len(parts) >= 2:
            return parts[0].strip(), parts[1].strip()
        return None, None

    return csv_blocks[0].strip(), csv_blocks[1].strip()

In [27]:
source_dir = Path("ocr-result")
files = sorted(list(source_dir.glob("**/*.txt")))
files

[WindowsPath('ocr-result/นอกเขต/ชุดที่ 1/ชุดที่ 1 ส.ส. 5-17(บช).txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 1/ชุดที่ 1 ส.ส. 5-17.txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 10/ชุดที่ 10 5-17(บช).txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 10/ชุดที่ 10 5-17.txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 11/ชุดที่ 11 5-17(บช).txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 11/ชุดที่ 11 5-17.txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 12/ชุดที่ 12 5-17 (บช).txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 12/ชุดที่ 12 5-17.txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 13/ชุดที่ 13 5-17 (บช).txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 13/ชุดที่ 13 5-17.txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 2/ชุดที่ 2 ส.ส. 5-17 (บช).txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 2/ชุดที่ 2 ส.ส. 5-17.txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 3/ชุดที่ 3 ส.ส. 5-17 (บช).txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 3/ชุดที่ 3 ส.ส. 5-17.txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 4/ชุดที่ 4 ส.ส.

In [28]:
def extract_file_info(filename):
    sub_district_match = re.search(r'(?:ตำบล|ต\.)\s*([\u0E00-\u0E7F]+)', filename)
    sub_district = sub_district_match.group(1) if sub_district_match else "-1"
    if sub_district_match:
        sub_district = sub_district_match.group(1)
    else:
        set_match = re.search(r'(ชุดที่\s*\d+)', filename)
        sub_district = set_match.group(1) if set_match else "-1"
    unit_match = re.search(r'หน่วยที่_?(\d+)', filename)
    unit_number = unit_match.group(1) if unit_match else "-1"

    return sub_district, unit_number

In [29]:
def remove_header(csv_text, header):
    if csv_text.startswith(header):
        return csv_text[len(header):].strip()
    return csv_text.strip()

In [30]:
with open("except_extract.txt", "r", encoding="utf-8") as fr:
    line_checks = [line.strip() for line in fr]

In [31]:
client = OpenAI(
    api_key=os.environ["TOKEN_KEY"],
    base_url="https://api.opentyphoon.ai/v1"
)

In [32]:
for file in files:
    try:
        if file.stem in line_checks:
            print(f"file: {file.stem} already extract!")
            struc = file.parent.parts
            print(f'อำเภอ: {struc[1]}')
            continue
        with open(file, 'r', encoding="utf-8") as f:
            lines = [line.strip() for line in f]
        content = "\n".join(lines) 
        structure = file.parent.parts
        sub_district, unit_num = extract_file_info(file.stem)
        if sub_district == "-1":
            sub_district = structure[2]
        csv1, csv2 = generate2csv(client, structure[1], sub_district, unit_num , province="ลำปาง", content=content, file_name=f"{file.stem}.txt")
        header1 = "type,province,district,sub-district,unit_number,name,score"
        header2 = "type,province,district,sub-district,unit_number,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots,remain_ballots"

        csv1_clean = remove_header(csv1, header1)
        csv2_clean = remove_header(csv2, header2)
        print(f"Starting save result of file: {file.stem}")
        with open(f"master_results.csv", "a", encoding="utf-8") as fw1:
            fw1.write(csv1_clean.strip() +"\n")
        with open(f"master_summary.csv", "a", encoding="utf-8") as fw2:
            fw2.write(csv2_clean.strip() +"\n")
        with open(f"except_extract.txt", "a", encoding="utf-8") as fw3:
            fw3.write(f"{file.stem}\n")
        print(f"Success save result on file: {file.stem}")
    except Exception as e:
        print(f"LLM error on {file.stem}: {e}")
        continue

file: ชุดที่ 1 ส.ส. 5-17(บช) already extract!
อำเภอ: นอกเขต
file: ชุดที่ 1 ส.ส. 5-17 already extract!
อำเภอ: นอกเขต
file: ชุดที่ 10 5-17(บช) already extract!
อำเภอ: นอกเขต
file: ชุดที่ 10 5-17 already extract!
อำเภอ: นอกเขต
file: ชุดที่ 11 5-17(บช) already extract!
อำเภอ: นอกเขต
file: ชุดที่ 11 5-17 already extract!
อำเภอ: นอกเขต
file: ชุดที่ 12 5-17 (บช) already extract!
อำเภอ: นอกเขต
file: ชุดที่ 12 5-17 already extract!
อำเภอ: นอกเขต
file: ชุดที่ 13 5-17 (บช) already extract!
อำเภอ: นอกเขต
file: ชุดที่ 13 5-17 already extract!
อำเภอ: นอกเขต
file: ชุดที่ 2 ส.ส. 5-17 (บช) already extract!
อำเภอ: นอกเขต
file: ชุดที่ 2 ส.ส. 5-17 already extract!
อำเภอ: นอกเขต
file: ชุดที่ 3 ส.ส. 5-17 (บช) already extract!
อำเภอ: นอกเขต
file: ชุดที่ 3 ส.ส. 5-17 already extract!
อำเภอ: นอกเขต
file: ชุดที่ 4 ส.ส. 5-17(บช) already extract!
อำเภอ: นอกเขต
file: ชุดที่ 4 ส.ส. 5-17 already extract!
อำเภอ: นอกเขต
file: ชุดที่ 5 ส.ส. 5-17(บช) already extract!
อำเภอ: นอกเขต
Processing CSVs: ชุดที่ 5 ส.ส. 5-17.txt
